In [6]:
import pandas as pd
import librosa
import soundfile as sf
import os
import glob


# CSVがあるフォルダ
csv_dir = "/home/mitani/CSJ-emo-int_bunseki/0718/0720/LR_bunseki/data/Scripts"

# WAVフォルダ
wav_dir = "/autofs/diamond/share/corpus/CSJ/WAV/noncore"

# 出力先
out_root = "data/Segments"
os.makedirs(out_root, exist_ok=True)


# CSV一覧取得
csv_files = glob.glob(
    os.path.join(csv_dir, "*.csv")
)


for csv_file in csv_files:

    # 例: D03F0001.csv
    base_name = os.path.splitext(
        os.path.basename(csv_file)
    )[0]

    print(f"processing {base_name}")


    # CSV読み込み
    df = pd.read_csv(csv_file)


    # 出力フォルダ
    out_dir = os.path.join(
        out_root,
        base_name
    )

    os.makedirs(
        out_dir,
        exist_ok=True
    )


    save_info = []


    # 話者ごとに処理
    for speaker in df["speaker"].unique():


        # D03F0001-L.wav
        wav_file = os.path.join(
            wav_dir,
            f"{base_name}-{speaker}.wav"
        )


        if not os.path.exists(wav_file):
            print(f"WAVなし: {wav_file}")
            continue


        # wav読み込み
        audio, sr = librosa.load(
            wav_file,
            sr=None
        )


        # 話者フォルダ
        speaker_dir = os.path.join(
            out_dir,
            speaker
        )

        os.makedirs(
            speaker_dir,
            exist_ok=True
        )


        # 話者発話のみ
        df_spk = df[df["speaker"] == speaker]


        for _, row in df_spk.iterrows():

            utt_id = row["utt_id"]
            start = row["start"]
            end = row["end"]
            text = row["text"]

            # <雑音>をスキップ
            if text == "<雑音>":
                continue

            # 秒→サンプル
            start_sample = int(start * sr)
            end_sample = int(end * sr)


            segment = audio[start_sample:end_sample]


            # wav名
            wav_name = f"{utt_id}_{speaker}.wav"


            wav_path = os.path.join(
                speaker_dir,
                wav_name
            )


            # 保存
            sf.write(
                wav_path,
                segment,
                sr
            )


            # CSV用情報
            save_info.append([
                wav_name,
                start,
                end,
                text
            ])


    # 発話情報CSV保存
    df_out = pd.DataFrame(
        save_info,
        columns=[
            "wavfile",
            "start",
            "end",
            "text"
        ]
    )


    df_out.to_csv(
        os.path.join(
            out_dir,
            f"{base_name}_segments.csv"
        ),
        index=False,
        encoding="utf-8"
    )


print("全CSV処理完了")

processing D03F0008
processing D03M0013
processing D03M0007
processing D03M0037
processing D03M0038
processing D03F0001
processing D03F0045
processing D03M0017
processing D03F0036
processing D03F0058
processing D03F0040
processing D03M0048
processing D03F0034
processing D03F0006
processing D03M0053
processing D03M0004
全CSV処理完了
